In [ ]:
import json
import uuid
from typing import Annotated, Any

from kaggle_benchmarks.tools import ToolInvocation
from typing_extensions import TypedDict

from kagraph import END, START, InMemorySaver, StateGraph, add_messages, invoke_llm
from kagraph.llms import load_llm
from kagraph.messages import AnyMessage, HumanMessage, ToolMessage
from kagraph.tracing import trace

In [ ]:
INFO_SYSTEM = """Your job is to gather requirements for a prompt template.

You need these fields:
- objective
- variables that will be passed into the prompt template
- constraints for what the output should NOT do
- requirements the output MUST follow

If any field is missing, ask the user one concise clarifying question.
If all fields are present, call the PromptInstructions tool. Do not guess missing fields.
"""

PROMPT_SYSTEM = """Based on the following requirements, write a high-quality prompt template.

Requirements:
{requirements}
"""

In [ ]:
def replace_latest(_left: Any, right: Any) -> Any:
    return right


class InformationInput(TypedDict):
    message: Annotated[str, replace_latest]


class InformationState(TypedDict, total=False):
    message: Annotated[str, replace_latest]
    messages: Annotated[list[AnyMessage], add_messages]
    prompt_requirements: dict[str, Any]
    generated_prompt: str

In [ ]:
def PromptInstructions(
    objective: str,
    variables: list[str],
    constraints: list[str],
    requirements: list[str],
) -> str:
    """Call when enough information has been gathered to generate the prompt."""

    return "Prompt requirements captured."


def _message_text(message: AnyMessage) -> str:
    text = getattr(message, "text", None)
    if text is not None:
        return str(text)
    return str(getattr(message, "content", ""))


def _tool_calls(message: AnyMessage) -> list[Any]:
    calls = getattr(message, "tool_calls", None)
    if calls is not None:
        return list(calls)
    return list((getattr(message, "_meta", {}) or {}).get("tool_calls") or [])


def _first_prompt_instruction_call(message: AnyMessage) -> ToolInvocation | None:
    calls = _tool_calls(message)
    if not calls:
        return None

    invocation = ToolInvocation.from_api_dict(calls[0])
    if invocation.name != "PromptInstructions":
        raise ValueError(f"Unexpected tool call {invocation.name!r}")
    return invocation


def _normalize_prompt_arguments(arguments: dict[str, Any]) -> dict[str, Any]:
    if "signature" in arguments and isinstance(arguments["signature"], dict):
        arguments = arguments["signature"]

    allowed = {"objective", "variables", "constraints", "requirements"}
    normalized = {key: value for key, value in dict(arguments).items() if key in allowed}
    for key in ("variables", "constraints", "requirements"):
        value = normalized.get(key)
        if isinstance(value, str):
            normalized[key] = [item.strip() for item in value.split(",") if item.strip()]

    missing = allowed - set(normalized)
    if missing:
        raise ValueError(
            "PromptInstructions tool call is missing required fields: "
            f"{sorted(missing)}"
        )
    return normalized


def build_graph(llm):
    graph = StateGraph(InformationState, input_schema=InformationInput)

    def prepare_user_message(state: InformationInput):
        return {"messages": [HumanMessage(state["message"])]}

    def collect_requirements(state: InformationState):
        response = invoke_llm(
            llm,
            messages=state["messages"],
            prompt=(
                "Continue the information-gathering conversation. Treat the latest "
                "user message as new information for the prompt requirements. If it "
                "partially answers a required field, acknowledge that and ask only "
                "for the next missing field."
            ),
            system=INFO_SYSTEM,
            tools=[PromptInstructions],
        )
        return {"messages": [response]}

    def capture_prompt_requirements(state: InformationState):
        invocation = _first_prompt_instruction_call(state["messages"][-1])
        if invocation is None:
            raise ValueError("Expected latest message to contain a PromptInstructions tool call.")

        arguments = _normalize_prompt_arguments(invocation.arguments)
        return {
            "messages": [
                ToolMessage(
                    PromptInstructions(**arguments),
                    name=invocation.name,
                    tool_call_id=invocation.call_id,
                )
            ],
            "prompt_requirements": arguments,
        }

    def generate_prompt(state: InformationState):
        requirements = state.get("prompt_requirements")
        if not requirements:
            raise ValueError("No prompt requirements found.")

        response = invoke_llm(
            llm,
            prompt="Generate the prompt template now. Include placeholders for all variables.",
            system=PROMPT_SYSTEM.format(
                requirements=json.dumps(requirements, ensure_ascii=False, indent=2)
            ),
        )
        return {"messages": [response], "generated_prompt": response.text}

    def route_after_collection(state: InformationState):
        return "capture_prompt_requirements" if _tool_calls(state["messages"][-1]) else END

    graph.add_node("prepare_user_message", prepare_user_message, input_schema=InformationInput)
    graph.add_node("collect_requirements", collect_requirements)
    graph.add_node("capture_prompt_requirements", capture_prompt_requirements)
    graph.add_node("generate_prompt", generate_prompt)

    graph.add_edge(START, "prepare_user_message")
    graph.add_edge("prepare_user_message", "collect_requirements")
    graph.add_conditional_edges(
        "collect_requirements",
        route_after_collection,
        {
            "capture_prompt_requirements": "capture_prompt_requirements",
            END: END,
        },
    )
    graph.add_edge("capture_prompt_requirements", "generate_prompt")
    graph.add_edge("generate_prompt", END)

    return graph.compile(
        name="information_gather_prompting",
        checkpointer=InMemorySaver(),
    )

In [ ]:
llm = load_llm("qwen/qwen3-next-80b-a3b-instruct")
app = build_graph(llm)

In [ ]:
app.get_graph().draw_png()

In [ ]:
config = {"configurable": {"thread_id": str(uuid.uuid4())}}
print("Interactive prompt generator. Type q to quit.")
while True:
    user_message = input("\nUser: ").strip()
    if user_message.lower() == "q":
        print("AI: Byebye")
        break
    if not user_message:
        continue
    
    result = app.invoke(
        {"message": user_message},
        config=config,
        chat_name="KaGraph Information Gather Prompting tutorial",
        recursion_limit=10,
    )
    last_message = result["messages"][-1]
    print(f"\nAI: {_message_text(last_message)}")
    if result.get("generated_prompt"):
        print("\nGenerated prompt:\n")
        print(result["generated_prompt"])
        break